# 10 — Bukti Kuantitatif Distribution Shift: Wasserstein Distance pada 9 Fitur SFM

**Tujuan (menjawab reviewer Q1):** membuktikan secara *kuantitatif dan matematis* — bukan sekadar visual atau kenaikan MCC — bahwa adaptasi domain berbiaya-rendah (few-shot 1% label target, dan cross-dataset mixup) **memperkecil jarak statistik** antara distribusi fitur yang "dilihat" model saat latih dan distribusi jaringan **target**. Metrik: **Wasserstein Distance (Earth Mover's Distance)**, mengikuti standar Layeghy dkk. (2024).

**Protokol (jujur, konsisten dengan notebook 06):**
- 9 fitur SFM (Model A), pemetaan `MAP_A` identik dengan notebook 03–06.
- Semua distribusi diproyeksikan ke **ruang z-score jaringan target** (scaler di-*fit* pada target-train). Ini penting: jarak Wasserstein dihitung pada skala yang sama sehingga bermakna sebagai ukuran *shift* relatif terhadap target.
- Untuk setiap fitur $j$ kami hitung $W_1$ antara **distribusi target-test** (acuan tetap) dan distribusi **himpunan latih** pada tiga kondisi:
  1. **Sebelum kalibrasi** — latih = *source* saja (baseline single-source, T3).
  2. **Few-shot 1%** — latih = *source* + 1% label *target-train*.
  3. **Mixup** — latih = *source* + sampel *mixup*(source, target-train), tanpa label target-test.
- Ringkasan: rata-rata $\overline{W}_1$ atas 9 fitur (dan per-fitur). Dua arah: CIC→UNSW dan UNSW→CIC.

**Hipotesis jujur:** $\overline{W}_1$ turun dari kondisi *sebelum kalibrasi* → *few-shot/mixup*. Jika turun, ini bukti kuantitatif bahwa kalibrasi menyelaraskan distribusi ke target. Jika tidak turun, dilaporkan apa adanya. **Tidak ada angka karangan** — seluruh nilai ditulis oleh notebook ini ke `wasserstein_shift.json`.

> Jalankan di SageMaker (butuh `cleaned_100.pkl` + CSV UNSW), sama seperti notebook 03–08.

In [ ]:
# --- Bootstrap ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('numpy','numpy'), ('scikit-learn','sklearn'), ('scipy','scipy')]:
    try: importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...'); subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import pickle, os, json
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import wasserstein_distance

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
# CATATAN: nama file UNSW tertukar (testing-set=train 175k; training-set=test 82k) — sama dgn notebook 06.
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
OUT_JSON='../wasserstein_shift.json'
SEED=42
FEWSHOT_FRAC=0.01   # 1% label target (konsisten dgn temuan kalibrasi minimal di notebook 06)
print('CIC:', os.path.exists(CIC_PKL), '| UNSW tr:', os.path.exists(UNSW_TRAIN), '| te:', os.path.exists(UNSW_TEST))

In [ ]:
# --- Pemetaan 9 fitur SFM (identik notebook 03–06) ---
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys())

def build_matrix(df, side):
    idx=0 if side=='cic' else 1
    cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON
    out=out.replace([np.inf,-np.inf],np.nan)
    out=out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values

def mixup(Xs, ys, Xt, yt, n_mix, alpha=0.4):
    """identik notebook 06: x = lam*src + (1-lam)*tgt_train; label = komponen dominan."""
    rng=np.random.RandomState(SEED)
    si=rng.choice(len(Xs), n_mix, replace=True)
    ti=rng.choice(len(Xt), n_mix, replace=True)
    lam=rng.beta(alpha, alpha, size=(n_mix,1))
    Xmix=lam*Xs[si]+(1-lam)*Xt[ti]
    ymix=np.where(lam.ravel()>=0.5, ys[si], yt[ti])
    return Xmix, ymix.astype(int)

def avg_wasserstein(X_ref, X_cmp, feat_names):
    """W1 per-fitur antara X_ref (acuan target) dan X_cmp (himpunan latih), lalu rata-rata."""
    per={}
    for j,name in enumerate(feat_names):
        per[name]=float(wasserstein_distance(X_ref[:,j], X_cmp[:,j]))
    per_mean=float(np.mean(list(per.values())))
    return per_mean, per

In [ ]:
# --- Muat data (identik notebook 06) ---
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float)
sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0)
y_cic=(np.asarray(d['y'])!=benign).astype(int)

unsw_tr=pd.read_csv(UNSW_TRAIN); unsw_te=pd.read_csv(UNSW_TEST)
y_utr=unsw_tr['label'].astype(int).values; y_ute=unsw_te['label'].astype(int).values

Xc_all=build_matrix(cic_df,'cic')
Xc_tr_raw,Xc_te_raw,yc_tr,yc_te=train_test_split(Xc_all,y_cic,test_size=0.3,random_state=SEED,stratify=y_cic)
Xu_tr_raw=build_matrix(unsw_tr,'unsw'); Xu_te_raw=build_matrix(unsw_te,'unsw')
print('CIC tr/te raw:',Xc_tr_raw.shape,Xc_te_raw.shape,'| UNSW tr/te raw:',Xu_tr_raw.shape,Xu_te_raw.shape)

In [ ]:
# ============================================================
# Hitung Wasserstein untuk satu arah source->target
#   Acuan tetap: target-TEST (jaringan yang ingin dilayani).
#   Dibandingkan thd himpunan LATIH pada 3 kondisi.
#   Semua di ruang z-score TARGET (scaler fit di target-train) agar sebanding.
# ============================================================
def run_direction(Xs_raw, ys, Xt_tr_raw, yt_tr, Xt_te_raw, tag):
    print('='*70); print(f'ARAH: {tag}'); print('='*70)
    sc_t=StandardScaler().fit(Xt_tr_raw)              # ruang acuan = TARGET
    Xs=sc_t.transform(Xs_raw)                          # source di ruang target
    Xt_tr=sc_t.transform(Xt_tr_raw); Xt_te=sc_t.transform(Xt_te_raw)

    # (1) sebelum kalibrasi: latih = source saja
    m0,per0=avg_wasserstein(Xt_te, Xs, CANON)

    # (2) few-shot 1%: latih = source + 1% label target-train
    rng=np.random.RandomState(SEED)
    n=int(len(Xt_tr)*FEWSHOT_FRAC); idx=rng.choice(len(Xt_tr), n, replace=False)
    X_fs=np.vstack([Xs, Xt_tr[idx]])
    m1,per1=avg_wasserstein(Xt_te, X_fs, CANON)

    # (3) mixup: latih = source + mixup(source, target-train)
    n_mix=min(len(Xs), 100000)
    Xmix,_=mixup(Xs,ys, Xt_tr,yt_tr, n_mix)
    X_mx=np.vstack([Xs, Xmix])
    m2,per2=avg_wasserstein(Xt_te, X_mx, CANON)

    # referensi batas-bawah: target-train vs target-test (shift intra-domain)
    m_lb,per_lb=avg_wasserstein(Xt_te, Xt_tr, CANON)

    print(f'  W1 rata-rata (9 fitur):')
    print(f'    sebelum kalibrasi (source only) : {m0:.4f}')
    print(f'    few-shot 1% label target        : {m1:.4f}  (reduksi {100*(m0-m1)/m0:+.1f}%)')
    print(f'    mixup (tanpa label target-test) : {m2:.4f}  (reduksi {100*(m0-m2)/m0:+.1f}%)')
    print(f'    [acuan batas-bawah] target-train: {m_lb:.4f}')
    return dict(
        before=dict(mean=m0, per_feature=per0),
        fewshot_1pct=dict(mean=m1, per_feature=per1, reduction_pct=100*(m0-m1)/m0),
        mixup=dict(mean=m2, per_feature=per2, reduction_pct=100*(m0-m2)/m0),
        target_train_lb=dict(mean=m_lb, per_feature=per_lb),
    )

results={}
results['cic2unsw']=run_direction(Xc_tr_raw,yc_tr, Xu_tr_raw,y_utr, Xu_te_raw, 'CIC(source) -> UNSW(target)')
results['unsw2cic']=run_direction(Xu_tr_raw,y_utr, Xc_tr_raw,yc_tr, Xc_te_raw, 'UNSW(source) -> CIC(target)')

In [ ]:
# --- Ringkasan + simpan JSON (angka nyata, untuk mengisi tabel di paper) ---
print('\nRINGKASAN W1 RATA-RATA (9 fitur SFM, ruang z-score target):')
print(f"{'arah':>12} {'sebelum':>10} {'few-shot1%':>12} {'mixup':>10}")
for k,lab in [('cic2unsw','CIC->UNSW'),('unsw2cic','UNSW->CIC')]:
    r=results[k]
    print(f"{lab:>12} {r['before']['mean']:>10.4f} {r['fewshot_1pct']['mean']:>12.4f} {r['mixup']['mean']:>10.4f}")

meta=dict(
  deskripsi='Wasserstein Distance (W1) per-fitur & rata-rata pada 9 fitur SFM, dihitung di ruang z-score target. '
            'Membuktikan kuantitatif bahwa kalibrasi domain (few-shot 1% / mixup) memperkecil jarak ke distribusi target.',
  metrik='scipy.stats.wasserstein_distance (Earth Mover distance, W1) per fitur, lalu rata-rata 9 fitur.',
  ruang='z-score dengan StandardScaler fit pada target-train (acuan target).',
  acuan='target-test (tetap); dibandingkan thd himpunan latih pd 3 kondisi + batas-bawah target-train.',
  features=CANON, fewshot_frac=FEWSHOT_FRAC, seed=SEED,
  results=results,
)
with open(OUT_JSON,'w') as f: json.dump(meta,f,indent=2)
print('\nSaved:', OUT_JSON)